In [19]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
import gradio as gr

In [4]:
load_dotenv(override=True)
MODEL = "gpt-5-nano"
DB_NAME = "vector_db"

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [6]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model=MODEL)

In [7]:
retriever.invoke("Who is Kim?")

[Document(id='89c1584b-32bc-424c-bdec-66f8b67f96bc', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\David Kim.md'}, page_content='# HR Record\n\n# David Kim\n\n## Summary\n- **Date of Birth:** September 22, 1992\n- **Job Title:** DevOps Engineer\n- **Location:** New York, New York\n- **Current Salary:** $118,000\n\n## Insurellm Career Progression\n- **August 2021 - Present:** DevOps Engineer\n  - Manages AWS infrastructure for all Insurellm products\n  - Implemented CI/CD pipelines reducing deployment time by 60%\n  - Led migration to Kubernetes, improving system scalability and reliability\n\n- **March 2019 - July 2021:** Junior DevOps Engineer\n  - Assisted with infrastructure monitoring and incident response\n  - Automated manual deployment processes using Jenkins and Terraform\n  - Supported senior engineers in cloud infrastructure management\n\n- **June 2017 - February 2019:** Systems Administrator at CloudFirst Solutions\n  - Maintained Linux servers and

In [8]:
llm.invoke("Who is Kim?")

AIMessage(content='“That could refer to many people. Do you mean a specific Kim? If you can share a bit of context (country, field, or a clue), I can identify or describe the right person.\n\nHere are a few common “Kim” references to help pin it down:\n- Kim Kardashian (American media personality, entrepreneur)\n- Kim Jong-un (North Korea’s supreme leader)\n- Kim Il-sung (founding leader of North Korea)\n- Kim Basinger, Kim Cattrall, Kim Novak (well-known actresses)\n- Kim Yu-na (Yuna Kim, South Korean Olympic figure skater)\n- Kim Possible (fictional character from the animated series)\n- In Korea, Kim is the most common surname (written Kim, meaning 金 “gold/metal”)\n\nIf you tell me a bit more about which Kim you have in mind, I can give a short bio or key facts about that person.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1090, 'prompt_tokens': 10, 'total_tokens': 1100, 'completion_tokens_details': {'accepted_prediction_tokens': 0,

In [9]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgable, friendly assistant representing the compnay InsureLLM.
You are chatting with user about InsureLLM.
If relevant, use the below context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [32]:
def combined_question(question, history):
    prior = "\n".join(h["content"] for h in history if h["role"] == "user")
    return prior + "\n" + question

In [33]:
def answer_question(question: str, history):
    combined = combined_question(question=question, history=history)
    docs = retriever.invoke(combined)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)

    return response.content

In [22]:
print(answer_question("Who is Kim?", []))

Kim refers to David Kim. He is a DevOps Engineer at InsureLLM, based in New York, born September 22, 1992, with a current salary of $118,000.

Key highlights:
- Since August 2021: Manages AWS infrastructure for all InsureLLM products.
- Implemented CI/CD pipelines, cutting deployment time by about 60%.
- Led the migration to Kubernetes to improve system scalability and reliability.

If you’d like more details from his HR record, I can pull them up.


In [34]:
gr.ChatInterface(fn=answer_question, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
